# Eksploracyjna Analiza Danych — Nieruchomości w Rumunii
**Źródło danych:** imobiliare.ro  
**Autor:** Yuliia Kuliievych
**Opis:** Analiza ogłoszeń mieszkań na sprzedaż zebranych przez scraper z portalu imobiliare.ro.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

df = pd.read_csv('../data/processed/imobiliare.csv', encoding='utf-8-sig')
print(f"Wymiary zbioru: {df.shape}")
df.head()

## 1. Podstawowe informacje o zbiorze

In [ ]:
print("Typy kolumn:")
print(df.dtypes)
print()
print("Brakujące wartości:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df.describe()[['price_eur', 'area_m2', 'rooms', 'desc_length']].round(2)

## 2. Rozkład ceny

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Surowy rozkład
axes[0].hist(df['price_eur'], bins=50, color='#838F58', edgecolor='white')
axes[0].set_title('Rozkład ceny (surowy)')
axes[0].set_xlabel('Cena (EUR)')
axes[0].set_ylabel('Liczba ogłoszeń')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Logarytmiczny
axes[1].hist(np.log1p(df['price_eur']), bins=50, color='#DB918F', edgecolor='white')
axes[1].set_title('Rozkład ceny (log)')
axes[1].set_xlabel('log(Cena + 1)')
axes[1].set_ylabel('Liczba ogłoszeń')

plt.tight_layout()
plt.show()
print(f"Mediana: {df['price_eur'].median():,.0f} EUR")
print(f"Średnia: {df['price_eur'].mean():,.0f} EUR")
print(f"Odch. std: {df['price_eur'].std():,.0f} EUR")

## 3. Rozkład powierzchni

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_filtered = df[df['area_m2'] <= 300]

axes[0].hist(df_filtered['area_m2'], bins=50, color='#5A86CB', edgecolor='white')
axes[0].set_title('Rozkład powierzchni (≤300 m²)')
axes[0].set_xlabel('Powierzchnia (m²)')
axes[0].set_ylabel('Liczba ogłoszeń')

axes[1].scatter(df_filtered['area_m2'], df_filtered['price_eur'],
                alpha=0.3, color='#838F58', s=10)
axes[1].set_title('Powierzchnia vs Cena')
axes[1].set_xlabel('Powierzchnia (m²)')
axes[1].set_ylabel('Cena (EUR)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 4. Analiza kategorii i układów

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kategorie
cat_counts = df['category'].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color='#F2AEBC')
axes[0].set_title('Liczba ogłoszeń według kategorii')
axes[0].set_xlabel('Liczba')

# Układ (layout)
layout_cols = ['layout_garsoniera', 'layout_nedecomandat', 'layout_nedefinit',
               'layout_penthouse', 'layout_semidecomandat']
layout_labels = ['Kawalerka', 'Przechodny', 'Nieokreślony', 'Penthouse', 'Półrozdzielny']
layout_counts = df[layout_cols].sum()
other = len(df) - layout_counts.sum()

labels = layout_labels + ['Rozdzielny']
values = list(layout_counts.values) + [other]
axes[1].barh(labels, values, color='#5A86CB')
axes[1].set_title('Typ układu mieszkania')
axes[1].set_xlabel('Liczba ogłoszeń')

plt.tight_layout()
plt.show()

## 5. Cena według liczby pokoi

In [ ]:
df['rooms'] = pd.to_numeric(df['rooms'], errors='coerce')
df_rooms = df[df['rooms'].between(1, 6)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
df_rooms.boxplot(column='price_eur', by='rooms', ax=axes[0])
axes[0].set_title('Cena według liczby pokoi')
axes[0].set_xlabel('Liczba pokoi')
axes[0].set_ylabel('Cena (EUR)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
plt.sca(axes[0])
plt.title('Cena według liczby pokoi')

# Średnia cena
mean_price = df_rooms.groupby('rooms')['price_eur'].median()
axes[1].bar(mean_price.index, mean_price.values, color='#838F58')
axes[1].set_title('Mediana ceny według liczby pokoi')
axes[1].set_xlabel('Liczba pokoi')
axes[1].set_ylabel('Mediana ceny (EUR)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 6. Top 10 lokalizacji według liczby ogłoszeń i mediany ceny

In [ ]:
top_localities = df['locality'].value_counts().head(10).index
df_top = df[df['locality'].isin(top_localities)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Liczba ogłoszeń
counts = df['locality'].value_counts().head(10)
axes[0].barh(counts.index[::-1], counts.values[::-1], color='#F2AEBC')
axes[0].set_title('Top 10 lokalizacji — liczba ogłoszeń')
axes[0].set_xlabel('Liczba ogłoszeń')

# Mediana ceny
median_price = df_top.groupby('locality')['price_eur'].median().reindex(top_localities)
axes[1].barh(median_price.index[::-1], median_price.values[::-1], color='#5A86CB')
axes[1].set_title('Top 10 lokalizacji — mediana ceny')
axes[1].set_xlabel('Mediana ceny (EUR)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 7. Udogodnienia — częstość i wpływ na cenę

In [ ]:
amenities = ['has_parking', 'has_balcony', 'has_elevator', 'has_pool',
             'has_metro', 'has_ac', 'has_terrace', 'has_garden',
             'has_new_building', 'has_own_heating']

amenity_labels = ['Parking', 'Balkon', 'Winda', 'Basen',
                  'Metro', 'Klimatyzacja', 'Taras', 'Ogród',
                  'Nowy budynek', 'Własne ogrzewanie']

df[amenities] = df[amenities].apply(pd.to_numeric, errors='coerce')
freq = df[amenities].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(amenity_labels[::-1], freq.values[::-1], color='#838F58')
axes[0].set_title('Częstość udogodnień (%)')
axes[0].set_xlabel('% ogłoszeń')

# Wpływ na cenę
price_diff = []
for col in amenities:
    with_feat = df[df[col] == 1]['price_eur'].median()
    without_feat = df[df[col] == 0]['price_eur'].median()
    price_diff.append(with_feat - without_feat)

colors = ['#838F58' if x > 0 else '#DB918F' for x in price_diff]
axes[1].barh(amenity_labels[::-1], price_diff[::-1], color=colors[::-1])
axes[1].set_title('Różnica mediany ceny: z/bez udogodnienia (EUR)')
axes[1].set_xlabel('Różnica ceny (EUR)')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 8. Macierz korelacji

In [ ]:
numeric_cols = ['price_eur', 'area_m2', 'rooms', 'desc_length'] + amenities
df_corr = df[numeric_cols].apply(pd.to_numeric, errors='coerce').corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(df_corr, dtype=bool))
sns.heatmap(df_corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            xticklabels=['Cena', 'Pow.', 'Pokoje', 'Dł. opisu'] + amenity_labels,
            yticklabels=['Cena', 'Pow.', 'Pokoje', 'Dł. opisu'] + amenity_labels)
plt.title('Macierz korelacji cech')
plt.tight_layout()
plt.show()

## 9. Długość opisu a cena

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df['desc_length'] = pd.to_numeric(df['desc_length'], errors='coerce')
ax.scatter(df['desc_length'], df['price_eur'], alpha=0.2, color='#5A86CB', s=8)
ax.set_title('Długość opisu vs Cena')
ax.set_xlabel('Długość opisu (znaki)')
ax.set_ylabel('Cena (EUR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
plt.tight_layout()
plt.show()

corr = df[['desc_length', 'price_eur']].corr().iloc[0, 1]
print(f"Korelacja Pearsona: {corr:.3f}")

## 10. Wnioski

- **Cena** ma rozkład prawostronnie skośny — większość mieszkań kosztuje 50–200k EUR, ale są też outliersy powyżej 1M EUR
- **Powierzchnia** silnie koreluje z ceną (im większe mieszkanie, tym droższe)
- **Liczba pokoi** — mieszkania 3-pokojowe są najdroższe w medianie
- **Udogodnienia** — basen i taras najbardziej podnoszą cenę, własne ogrzewanie obniża (starsze budownictwo)
- **Lokalizacja** — Central i Pipera mają najwyższe mediany cen
- **Długość opisu** — słaba korelacja z ceną, ale dłuższe opisy często dotyczą droższych nieruchomości
